In [3]:
from datasets import load_dataset
import pandas as pd
import os

# 下载数据集
dataset = load_dataset("zzhdbw/Simplified_Chinese_Multi-Emotion_Dialogue_Dataset")

# 设置保存路径
save_dir = "chinese_emotion"  # 自定义路径
os.makedirs(save_dir, exist_ok=True)

# 保存为CSV文件
if isinstance(dataset, dict):
    # 如果数据集有多个split
    for split_name, split_data in dataset.items():
        file_path = os.path.join(save_dir, f"{split_name}.csv")
        split_data.to_pandas().to_csv(file_path, index=False, encoding='utf-8')
        print(f"{split_name} 已保存到: {file_path}")
else:
    # 如果只有一个数据集
    file_path = os.path.join(save_dir, "dataset.csv")
    dataset.to_pandas().to_csv(file_path, index=False, encoding='utf-8')
    print(f"数据集已保存到: {file_path}")

README.md:   0%|          | 0.00/793 [00:00<?, ?B/s]

(…)inese_Multi-Emotion_Dialogue_Dataset.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4159 [00:00<?, ? examples/s]

train 已保存到: chinese_emotion\train.csv


In [1]:
import pandas as pd

df = pd.read_parquet("hf://datasets/dair-ai/emotion/unsplit/train-00000-of-00001.parquet")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
import torch
import accelerate
import pandas as pd
import os
import shutil
import numpy as np
from sklearn.model_selection import train_test_split

print(f"Accelerate 版本: {accelerate.__version__}")
print(f"PyTorch 版本: {torch.__version__}")

# 定义8种情绪映射
emotion_mapping = {
    "开心": 0, "平静": 1, "伤心": 2, "生气": 3, 
    "惊讶": 4, "疑问": 5, "厌恶": 6, "关心":7,
}

id_to_emotion = {v: k for k, v in emotion_mapping.items()}

print("🎭 情绪类别映射:")
for emotion, idx in emotion_mapping.items():
    print(f"  {emotion} -> {idx}")

# 使用正确的模型路径
model_path = "model/hfl_chinese-roberta-wwm-ext/models--hfl--chinese-roberta-wwm-ext/snapshots/5c58d0b8ec1d9014354d691c538661bf00bfdb44"
print(f"📁 使用模型路径: {model_path}")

# 首先检查CSV文件结构
print("检查CSV文件结构...")
df = pd.read_csv("chinese_emotion/train.csv")
print(f"数据行数: {len(df)}")
print(f"列名: {df.columns.tolist()}")
print("\n标签分布:")
label_counts = df['label'].value_counts()
print(label_counts)
print("\n数据样例:")
print(df.head())

# 计算每个类别的样本数量
print(f"\n📊 详细数据分布:")
for emotion, count in label_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  {emotion}: {count} 条 ({percentage:.1f}%)")

# 检查数据平衡性
min_count = label_counts.min()
max_count = label_counts.max()
imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
print(f"\n不平衡比例: {imbalance_ratio:.2f}x")

if imbalance_ratio > 3:
    print("⚠️  数据不平衡，将调整类别权重")

# 检查情绪标签是否都在定义的情绪中
unique_labels = df['label'].unique()
print(f"\n发现的情绪标签: {unique_labels}")
for label in unique_labels:
    if label not in emotion_mapping:
        print(f"⚠️  警告: 标签 '{label}' 不在预定义的情绪映射中")

# 将文本标签转换为数字ID
df['label_id'] = df['label'].map(emotion_mapping)

# 检查是否有未映射的标签
if df['label_id'].isna().any():
    print("❌ 错误: 存在未映射的情绪标签，请检查数据")
    unmapped = df[df['label_id'].isna()]
    print("未映射的数据:")
    print(unmapped[['text', 'label']])
    df = df.dropna(subset=['label_id'])
    print(f"删除未映射行后数据量: {len(df)}")

# 分割训练集和验证集
print("\n📁 分割数据集...")
train_df, eval_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label_id'])  # 减少验证集比例

print(f"训练集: {len(train_df)} 条")
print(f"验证集: {len(eval_df)} 条")

# 保存处理后的数据到临时文件
train_csv = "temp_train_data.csv"
eval_csv = "temp_eval_data.csv"
train_df[['text', 'label_id']].to_csv(train_csv, index=False)
eval_df[['text', 'label_id']].to_csv(eval_csv, index=False)

print(f"保存训练数据到: {train_csv}")
print(f"保存验证数据到: {eval_csv}")

# 加载模型和分词器
print("\n正在从本地加载模型和分词器...")
try:
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, 
        local_files_only=True,
        num_labels=len(emotion_mapping),
        id2label=id_to_emotion,
        label2id=emotion_mapping,
        # 针对中等数据量优化模型配置
        attention_probs_dropout_prob=0.1,
        hidden_dropout_prob=0.1,
        classifier_dropout=0.1
    )
    print("✅ 模型和分词器从本地加载完成！")
except Exception as e:
    print(f"❌ 从本地加载失败: {e}")
    exit()

# 加载数据集
print("正在加载数据集...")
dataset = load_dataset("csv", data_files={
    "train": train_csv,
    "eval": eval_csv
})

# 数据预处理函数
def preprocess_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors=None
    )
    tokenized["labels"] = [int(label) for label in examples["label_id"]]
    return tokenized

# 对数据集进行编码
print("预处理数据...")
encoded_dataset = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=1000,  # 增加批处理大小加速处理
    remove_columns=dataset["train"].column_names
)

print(f"训练集样本数: {len(encoded_dataset['train'])}")
print(f"验证集样本数: {len(encoded_dataset['eval'])}")

# 针对4000+数据优化的训练参数
training_args = TrainingArguments(
       output_dir="./temp_training_output",
    
    # 核心训练参数
    per_device_train_batch_size=32,  # 针对4000+数据优化
    per_device_eval_batch_size=32,
    num_train_epochs=15,  # 增加训练轮次
    
    # 学习率和优化
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    
    # 日志和保存
    logging_steps=50,
    eval_steps=200,  # 评估频率
    save_steps=500,  # 保存频率
    
    # 评估设置
    # evaluation_strategy="steps",  # 按步数评估
    # save_strategy="steps",        # 按步数保存
    
    # # 其他设置
    # save_total_limit=2,
    # load_best_model_at_end=True,
    # metric_for_best_model="eval_loss",
    # greater_is_better=False,
    # report_to=None,
    # remove_unused_columns=False,
    # dataloader_pin_memory=False,
)

# 如果参数名有问题，使用这个兼容版本：
try:
    # 尝试新版本参数名
    training_args = TrainingArguments(
        output_dir="./temp_training_output",
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=12,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        logging_steps=50,
        logging_dir="./temp_logs",
        report_to=None,
        remove_unused_columns=False,
        dataloader_pin_memory=False,
    )
except:
    # 使用最稳定版本
    training_args = TrainingArguments(
        output_dir="./temp_training_output",
        per_device_train_batch_size=32,
        num_train_epochs=12,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_steps=50,
        save_steps=500,
        save_total_limit=2,
        report_to=None,
        remove_unused_columns=False,
    )

# 创建训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["eval"],
    tokenizer=tokenizer,
)

print("\n🚀 开始优化训练（针对4000+数据）...")
print(f"训练轮次: {training_args.num_train_epochs}")
print(f"批量大小: {training_args.per_device_train_batch_size}")
print(f"学习率: {training_args.learning_rate}")
print(f"训练集大小: {len(encoded_dataset['train'])}")
print(f"验证集大小: {len(encoded_dataset['eval'])}")
print(f"总步数估计: {len(encoded_dataset['train']) // training_args.per_device_train_batch_size * training_args.num_train_epochs}")

# 训练模型
print("\n开始训练...")
train_results = trainer.train()

# 保存最佳模型
final_model_dir = "my_final_model/my_model_3"
print(f"\n💾 保存最佳模型到: {final_model_dir}")
os.makedirs(final_model_dir, exist_ok=True)
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print("✅ 最佳模型保存完成！")



In [5]:
# 最终评估
eval_results = trainer.evaluate()
print(f"📊 验证集损失: {eval_results['eval_loss']:.4f}")

# 清理临时文件
for temp_file in [train_csv, eval_csv]:
    if os.path.exists(temp_file):
        os.remove(temp_file)

if os.path.exists("./temp_training_output"):
    shutil.rmtree("./temp_training_output")

print(f"\n🎉 训练完成！")

📊 验证集损失: 0.4851

🎉 训练完成！


In [ ]:
# 详细评估
print("\n📊 模型性能评估...")
eval_results = trainer.evaluate()
print(f"验证集损失: {eval_results['eval_loss']:.4f}")

# 如果有准确率指标
if 'eval_accuracy' in eval_results:
    print(f"验证集准确率: {eval_results['eval_accuracy']:.4f}")

# 预测示例测试
print("\n🧪 模型测试示例:")
test_texts = [
    "太多的事，慢慢地就不能做了，太多的人，渐渐地就不见了。原来，成长就注定是一个要丢失的过程。",
    "你问我为何时常沉默，有的人无话可说，有的话无人可说", 
    "我的目光化作一件无形的衣裳，悄悄披在你肩上，生怕世间的风雨将你吹凉。那是悄无声息的守望，如月光浸润大地。",
    "相信奇迹的人，本身跟奇迹一样了不起",
    "仿佛有什么不洁的东西黏着在感官上，像目睹了玫瑰的糜烂。一种源自心底的疏离，想将这一切从视野里彻底擦去。",
    "我的不耐烦应该已经表现得很明显了",
    "胸腔里仿佛困着一头咆哮的兽，灼热的气息炙烤着理智的弦。怒火是无声的雷鸣，在压抑的乌云里翻滚。",
]

model.eval()
for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = predictions.argmax().item()
    confidence = predictions[0][predicted_class].item()
    emotion = id_to_emotion[predicted_class]
    print(f"  '{text}' -> {emotion} ({confidence:.2%})")

# 清理临时文件
print("\n🧹 清理临时文件...")
for temp_file in [train_csv, eval_csv]:
    if os.path.exists(temp_file):
        os.remove(temp_file)
        print(f"删除临时文件: {temp_file}")

if os.path.exists("./temp_training_output"):
    shutil.rmtree("./temp_training_output")
    print("删除临时训练输出目录")

if os.path.exists("./temp_logs"):
    shutil.rmtree("./temp_logs")
    print("删除临时日志目录")

print(f"\n🎉 训练完成！")
print(f"💾 模型保存在: {final_model_dir}")
print(f"📈 最终验证损失: {eval_results['eval_loss']:.4f}")